In [1]:
from langchain_classic import LlamaCpp

In [2]:
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [4]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

We got no output as Phi-3 requires a specific prompt template.

Lets create a prompt template adhering Phi-3's expectation, and utilize the llm model to create our first chain. 

In [3]:
from langchain_classic import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [4]:
basic_chain = prompt | llm

In [7]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

In [ ]:
Now the chain is set up to take an input prompt and pass it to the LLM for processing. 
We don't need to worry about the formatting of the prompt every time we use LLM.

Multiple Chains

In [5]:
from langchain_classic import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

C:\Users\rosha\AppData\Local\Temp\ipykernel_23756\1767282371.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [9]:
title.invoke({"summary": "a girl that lost her mother"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of a Farewell: The Journey Through Loss"'}

In [6]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [7]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [8]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [10]:
llm_chain.invoke("a musician who has become war victim but still chose to spread love with his music after the war was over.")

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(
c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'summary': 'a musician who has become war victim but still chose to spread love with his music after the war was over.',
 'title': ' "Melodies of Resilience: A War Survivor\'s Journey to Heal with Harmonious Love"',
 'character': ' "Melodies of Resilience" tells the poignant tale of a passionate musician who, after enduring unimaginable wartime horrors, chooses to channel his pain into creating soul-stirring melodies that embody hope and unity. His hauntingly beautiful compositions become beacons of love and resilience in post-war society, inspiring healing through the universal language of music.',
 'story': ' In the heart-wrenching narrative "Melodies of Resilience: A War Survivor\'s Journey to Heal with Harmonious Love," we delve into the poignant life of an unnamed musician whose world was shattered by war. Amidst the ruins, our protagonist discovers solace in his violin strings, each note a testament to survival and an emblem of healing. The hauntingly beautiful melodies that eme

Memory

In [11]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


' Hello Maarten! The answer to 1 + 1 is 2.'

In [12]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


" I'm unable to determine your name as I don't have access to personal data of individuals. If you need assistance with something specific, feel free to ask!"

Stateless LLMs have no memory of previous conversation.

Conversation Buffer

In [13]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [15]:
from langchain_classic.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

C:\Users\rosha\AppData\Local\Temp\ipykernel_23756\3768838688.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history")


In [16]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'Hi! My name is Maarten. What is 1 + 1?',
 'chat_history': '',
 'text': " The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units in total.\n\nHere is the calculation:\n\n1 + 1 = 2"}

In [17]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

c:\Users\rosha\anaconda3\envs\test1\Lib\site-packages\llama_cpp\llama.py:1307: RuntimeWarning: Detected duplicate leading "<s>" in prompt, this will likely reduce response quality, consider removing it...
  warnings.warn(


{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten. What is 1 + 1?\nAI:  The answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another, resulting in two units in total.\n\nHere is the calculation:\n\n1 + 1 = 2",
 'text': " Your name is Maarten.\n\nI'm an AI and I don't have a personal name, but you can call me Assistant."}